In [84]:
import os
import cv2
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [85]:
import os

print(os.listdir("/kaggle/input"))

['datasets']


In [86]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets/adithashok/odir10k"):
    print(root)

/kaggle/input/datasets/adithashok/odir10k
/kaggle/input/datasets/adithashok/odir10k/val
/kaggle/input/datasets/adithashok/odir10k/val/diabetes
/kaggle/input/datasets/adithashok/odir10k/val/hypertension
/kaggle/input/datasets/adithashok/odir10k/val/ageDegeneration
/kaggle/input/datasets/adithashok/odir10k/val/glaucoma


KeyboardInterrupt: 

In [ ]:
import os

print(os.listdir("/kaggle/input/datasets/adithashok/odir10k/train"))
TRAIN_PATH = "/kaggle/input/datasets/adithashok/odir10k/train"

classes = sorted(os.listdir(TRAIN_PATH))

print(classes)

In [ ]:
import os

TRAIN_PATH = "/kaggle/input/datasets/adithashok/odir10k/train"

classes = sorted(os.listdir(TRAIN_PATH))

for cls in classes:

    class_folder = os.path.join(TRAIN_PATH, cls)

    count = len(os.listdir(class_folder))

    print(f"{cls}: {count}")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
def crop_black_border(img):

    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    thresh = cv2.threshold(
        gray,
        10,
        255,
        cv2.THRESH_BINARY
    )[1]

    coords = cv2.findNonZero(thresh)

    if coords is None:
        return img

    x,y,w,h = cv2.boundingRect(coords)

    crop = img[y:y+h, x:x+w]

    return crop

In [ ]:
def clahe_enhancement(img):

    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)

    l,a,b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    l = clahe.apply(l)

    merged = cv2.merge((l,a,b))

    enhanced = cv2.cvtColor(
        merged,
        cv2.COLOR_LAB2RGB
    )

    return enhanced

In [ ]:
def remove_background(img):

    blur = cv2.GaussianBlur(
        img,
        (0,0),
        sigmaX=10
    )

    enhanced = cv2.addWeighted(
        img,
        1.5,
        blur,
        -0.5,
        0
    )

    return enhanced

In [ ]:
IMG_SIZE = 300

def preprocess_image(path):

    img = cv2.imread(path)

    img = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    img = crop_black_border(img)

    img = clahe_enhancement(img)

    img = remove_background(img)

    img = cv2.resize(
        img,
        (IMG_SIZE, IMG_SIZE)
    )

    img = img.astype(np.float32)/255.0

    return img

In [ ]:
sample_path = "/kaggle/input/datasets/adithashok/odir10k/train/diabetes/" + \
              os.listdir("/kaggle/input/datasets/adithashok/odir10k/train/diabetes")[0]

img = preprocess_image(sample_path)

plt.figure(figsize=(6,6))
plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
original = cv2.imread(sample_path)
original = cv2.cvtColor(original,cv2.COLOR_BGR2RGB)

processed = preprocess_image(sample_path)

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.imshow(original)
plt.title("Original")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(processed)
plt.title("Processed")
plt.axis("off")

plt.show()

In [ ]:
img = cv2.imread(sample_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

crop = crop_black_border(img)

clahe_img = clahe_enhancement(crop)

final_img = remove_background(clahe_img)

plt.figure(figsize=(18,6))

plt.subplot(1,3,1)
plt.imshow(img)
plt.title("Original")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(clahe_img)
plt.title("CLAHE")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(final_img)
plt.title("CLAHE + Background Removal")
plt.axis("off")

plt.show()

In [ ]:
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.utils.class_weight import compute_class_weight

import numpy as np
TRAIN_DIR = "/kaggle/input/datasets/adithashok/odir10k/train"

VAL_DIR = "/kaggle/input/datasets/adithashok/odir10k/val"

TEST_DIR = "/kaggle/input/datasets/adithashok/odir10k/test"
IMG_SIZE = 300
BATCH_SIZE = 32

In [ ]:
train_datagen = ImageDataGenerator(

    rescale=1./255,

    rotation_range=20,

    zoom_range=0.15,

    width_shift_range=0.1,

    height_shift_range=0.1,

    brightness_range=[0.9,1.1],

    horizontal_flip=True,

    fill_mode='nearest'
)

In [ ]:
val_datagen = ImageDataGenerator(
    rescale=1./255
)
test_datagen = ImageDataGenerator(
    rescale=1./255
)

In [ ]:
val_generator = val_datagen.flow_from_directory(

    VAL_DIR,

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode='categorical',

    shuffle=False
)

In [ ]:
test_generator = test_datagen.flow_from_directory(

    TEST_DIR,

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode='categorical',

    shuffle=False
)

In [ ]:
print(train_generator.class_indices)

In [ ]:
classes = train_generator.classes

weights = compute_class_weight(

    class_weight='balanced',

    classes=np.unique(classes),

    y=classes
)

class_weights = dict(enumerate(weights))

print(class_weights)

In [ ]:
images, labels = next(train_generator)

plt.figure(figsize=(12,12))

for i in range(9):

    plt.subplot(3,3,i+1)

    plt.imshow(images[i])

    plt.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
print(train_generator.class_indices)
print(class_weights)

In [ ]:
from tensorflow.keras.applications.efficientnet import preprocess_input
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,

    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.9,1.1],
    horizontal_flip=True,
    fill_mode='nearest'
)
val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)
test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

In [ ]:
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy('mixed_float16')
from tensorflow.keras.applications import EfficientNetB3

from tensorflow.keras.layers import *

from tensorflow.keras.models import Model
base_model = EfficientNetB3(

    weights='imagenet',

    include_top=False,

    input_shape=(300,300,3)
)
base_model.trainable = False

In [ ]:
x = base_model.output

x = GlobalAveragePooling2D()(x)

x = BatchNormalization()(x)

x = Dropout(0.4)(x)

x = Dense(
    512,
    activation='relu'
)(x)

x = BatchNormalization()(x)

x = Dropout(0.3)(x)

outputs = Dense(
    7,
    activation='softmax',
    dtype='float32'
)(x)

model = Model(
    inputs=base_model.input,
    outputs=outputs
)

In [ ]:
import tensorflow as tf

model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),

    loss='categorical_crossentropy',

    metrics=['accuracy']
)
from tensorflow.keras.callbacks import *

callbacks = [

    ModelCheckpoint(

        "best_model.keras",

        save_best_only=True,

        monitor="val_accuracy",

        mode="max"
    ),

    ReduceLROnPlateau(

        monitor="val_loss",

        factor=0.3,

        patience=3,

        verbose=1
    ),

    EarlyStopping(

        monitor="val_loss",

        patience=8,

        restore_best_weights=True
    )
]

In [ ]:
model.summary()

In [ ]:
images, labels = next(train_generator)

print(images.min())
print(images.max())

print(images.shape)
print(labels.shape)

In [ ]:
images, labels = next(train_generator)

plt.figure(figsize=(12,12))

for i in range(9):

    plt.subplot(3,3,i+1)

    img = images[i]

    img = (img - img.min()) / (img.max() - img.min())

    plt.imshow(img)

    plt.axis("off")

plt.show()

In [ ]:
history = model.fit(

    train_generator,

    validation_data=val_generator,

    epochs=30,

    class_weight=class_weights,

    callbacks=callbacks
)

In [91]:
base_model.trainable = True

for layer in base_model.layers[:-50]:
    layer.trainable = False
    model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
    history_finetune = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)

Epoch 1/10


2026-06-24 16:41:41.394558: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-24 16:41:41.552685: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-24 16:41:41.703779: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-24 16:41:41.911244: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-24 16:41:42.102047: E external/local_xla/xla/stream_

KeyboardInterrupt: 

In [92]:
print(train_generator.samples)
print(val_generator.samples)
print(test_generator.samples)

8489
1818
1824
